# 04 — Tier 1 Evaluation EDA

Exploratory analysis of the Tier 1 stance-classification eval runs. Summarises accuracy and retrieval quality across the four ablation modes (baseline / semantic-only / BM25-only / hybrid) and across training-cutoff eras, then digs into the post-cutoff per-class confusion to see *how* each mode fails. Reads from the existing eval CSVs — does not re-run the evaluation.

Inputs:
- `data/eval_summary.csv` — aggregated per-mode / per-era metrics.
- `data/eval_runs/{baseline,semantic_only,bm25_only,hybrid}.csv` — per-meeting predictions.

Artifacts written to `figures/` at the repo root:
- `table1_ablation_summary.csv`
- `figure2_era_split.png`
- `figure3_postcutoff_by_mode.png`
- `table2_confusion_postcutoff.csv`

## 0. Setup
Imports, paths, and the shared color palette used across every figure.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = REPO_ROOT / 'data'
EVAL_RUNS_DIR = DATA_DIR / 'eval_runs'
FIG_DIR = REPO_ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

COLORS = {
    'baseline':  '#3E5779',  # dark slate blue
    'semantic':  '#7B8FA1',  # gray-blue
    'bm25':      '#C7A032',  # mustard / gold
    'hybrid':    '#1A2B4A',  # dark navy
}

DPI = 150
plt.rcParams.update({
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 100,
})

print('Repo root:', REPO_ROOT)
print('Figures will be saved to:', FIG_DIR)

## 1. Load eval summary
Load `data/eval_summary.csv` and pivot it into a per-mode / per-era view. Hit@5 is derived from `mean_precision_at_k` (with k=5 and exactly one relevant document per query, mean precision × 5 = hit rate).

In [ ]:
summary = pd.read_csv(DATA_DIR / 'eval_summary.csv')
summary['hit_at_5'] = summary['mean_precision_at_k'] * 5
summary

## 2. Ablation summary table
One row per mode, three eras × two metrics (accuracy and hit@5). The baseline row has no retrieval, so its hit@5 cells are `n/a`. The per-column best is bolded in the styled view to make winners pop out.

In [ ]:
MODE_ORDER = ['baseline', 'semantic_only', 'bm25_only', 'hybrid']
MODE_LABELS = {
    'baseline': 'Baseline (no retrieval)',
    'semantic_only': 'Semantic-only',
    'bm25_only': 'BM25-only',
    'hybrid': 'Hybrid (RRF)',
}

def get_metric(mode, era, col):
    row = summary[(summary['mode'] == mode) & (summary['training_era'] == era)].iloc[0]
    return row[col]

rows = []
for m in MODE_ORDER:
    rows.append({
        'Mode': MODE_LABELS[m],
        'Pre-cutoff Acc':  get_metric(m, 'pre_cutoff',  'accuracy'),
        'Pre-cutoff Hit@5': np.nan if m == 'baseline' else get_metric(m, 'pre_cutoff',  'hit_at_5'),
        'Post-cutoff Acc':  get_metric(m, 'post_cutoff', 'accuracy'),
        'Post-cutoff Hit@5': np.nan if m == 'baseline' else get_metric(m, 'post_cutoff', 'hit_at_5'),
        'Overall Acc':       get_metric(m, 'overall',     'accuracy'),
        'Overall Hit@5':     np.nan if m == 'baseline' else get_metric(m, 'overall',     'hit_at_5'),
    })

ablation = pd.DataFrame(rows)

# Save raw CSV (with 'n/a' where appropriate)
ablation_csv = ablation.copy()
for col in ['Pre-cutoff Hit@5', 'Post-cutoff Hit@5', 'Overall Hit@5']:
    ablation_csv[col] = ablation_csv[col].apply(lambda v: 'n/a' if pd.isna(v) else f'{v:.3f}')
for col in ['Pre-cutoff Acc', 'Post-cutoff Acc', 'Overall Acc']:
    ablation_csv[col] = ablation_csv[col].apply(lambda v: f'{v:.3f}')
ablation_csv.to_csv(FIG_DIR / 'table1_ablation_summary.csv', index=False)
print('Saved:', FIG_DIR / 'table1_ablation_summary.csv')

# Styled inline view: bold the per-column best (ignoring NaN)
metric_cols = [c for c in ablation.columns if c != 'Mode']

def highlight_max(s):
    is_max = s == s.max()
    return ['font-weight: bold' if v else '' for v in is_max]

def fmt(v):
    return 'n/a' if pd.isna(v) else f'{v:.3f}'

styled = (ablation.style
          .format({c: fmt for c in metric_cols})
          .apply(highlight_max, subset=metric_cols)
          .set_caption('Tier 1 stance ablation summary — per-column best in bold'))
styled

## 3. Era-split bar chart — baseline vs BM25
Two grouped bars per era. Pre-cutoff bars are equal (73.1% each); post-cutoff is where retrieval pays off (0.0% baseline → 91.7% BM25). This is the cleanest visual of the "training-cutoff problem" the BM25 retriever solves.

In [ ]:
eras = ['Pre-cutoff (n=26)', 'Post-cutoff (n=12)']
baseline_vals = [73.1, 0.0]
bm25_vals     = [73.1, 91.7]

x = np.arange(len(eras))
width = 0.36

fig, ax = plt.subplots(figsize=(7.0, 4.5))
b1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline (no retrieval)', color=COLORS['baseline'])
b2 = ax.bar(x + width/2, bm25_vals,     width, label='BM25-only',               color=COLORS['bm25'])

for bars in (b1, b2):
    for rect in bars:
        h = rect.get_height()
        ax.annotate(f'{h:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(eras)
ax.set_ylabel('Stance accuracy (%)')
ax.set_ylim(0, 100)
ax.legend(loc='upper left', frameon=False)
ax.yaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)

fig.tight_layout()
out = FIG_DIR / 'figure2_era_split.png'
fig.savefig(out, dpi=DPI, bbox_inches='tight')
print('Saved:', out)
plt.show()

## 4. Post-cutoff accuracy and hit@5 by mode
Four modes × two metrics on the post-cutoff slice. Accuracy is the headline number (large label on top of the bar); hit@5 sits as a smaller label inside the bar so we can read both off one chart. Baseline has no retrieval, so no hit@5 label.

In [ ]:
modes      = ['Baseline', 'Semantic only', 'BM25 only', 'Hybrid (RRF)']
mode_keys  = ['baseline', 'semantic',      'bm25',      'hybrid']
acc_vals   = [0.0,  58.3, 91.7, 75.0]
hit5_vals  = [None, 0.0,  92.0, 75.0]

x = np.arange(len(modes))
width = 0.55

fig, ax = plt.subplots(figsize=(8.5, 5.0))
bars = ax.bar(x, acc_vals, width, color=[COLORS[k] for k in mode_keys])

for rect, acc, h5 in zip(bars, acc_vals, hit5_vals):
    cx = rect.get_x() + rect.get_width() / 2
    ax.annotate(f'acc {acc:.1f}%', xy=(cx, acc), xytext=(0, 4),
                textcoords='offset points', ha='center', va='bottom',
                fontsize=11, fontweight='bold')
    if h5 is not None:
        # Place hit@5 label inside bar near the top; if bar too short, place above
        if acc >= 12:
            ax.annotate(f'hit@5 {h5:.0f}%', xy=(cx, acc - 4), ha='center', va='top',
                        fontsize=9, color='white')
        else:
            ax.annotate(f'hit@5 {h5:.0f}%', xy=(cx, acc), xytext=(0, 18),
                        textcoords='offset points', ha='center', va='bottom',
                        fontsize=9, color='dimgray')

ax.set_xticks(x)
ax.set_xticklabels(modes)
ax.set_ylabel('Post-cutoff stance accuracy (%)')
ax.set_ylim(0, 100)
ax.yaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)

fig.tight_layout()
out = FIG_DIR / 'figure3_postcutoff_by_mode.png'
fig.savefig(out, dpi=DPI, bbox_inches='tight')
print('Saved:', out)
plt.show()

## 5. Per-class confusion (post-cutoff, n=12)
Built from the per-row prediction CSVs filtered to `training_era == 'post_cutoff'`. For each mode, two rows (ground-truth Neutral, ground-truth Dovish) × four prediction buckets (Hawk / Neut / Dov / Unkn). Tells us *what kind of mistake* each mode makes, not just how often.

**Note.** The post-cutoff bucket has 0 hawkish meetings by construction (the Fed has been on hold or cutting since July 2023). Baseline's 'unknown' column is 12/12 refusals. BM25's only error is one dovish meeting predicted as neutral.

In [ ]:
STANCE_BUCKETS = ['hawkish', 'neutral', 'dovish', 'unknown']
BUCKET_LABELS  = {'hawkish': 'Hawk', 'neutral': 'Neut', 'dovish': 'Dov', 'unknown': 'Unkn'}
GT_ORDER       = ['neutral', 'dovish']  # post-cutoff has no hawkish ground truth

rows = []
for mode in MODE_ORDER:
    df = pd.read_csv(EVAL_RUNS_DIR / f'{mode}.csv')
    post = df[df['training_era'] == 'post_cutoff']
    for gt in GT_ORDER:
        sub = post[post['ground_truth_stance'] == gt]
        # Anything not in our 4 known buckets falls into 'unknown'
        pred = sub['predicted_stance'].where(sub['predicted_stance'].isin(STANCE_BUCKETS), 'unknown')
        counts = pred.value_counts().reindex(STANCE_BUCKETS, fill_value=0)
        rows.append({
            'Mode': MODE_LABELS[mode],
            'Ground truth': f"{gt.capitalize()} (n={len(sub)})",
            **{BUCKET_LABELS[b]: int(counts[b]) for b in STANCE_BUCKETS},
        })

confusion = pd.DataFrame(rows)
confusion.to_csv(FIG_DIR / 'table2_confusion_postcutoff.csv', index=False)
print('Saved:', FIG_DIR / 'table2_confusion_postcutoff.csv')

caption = ('Per-class confusion (post-cutoff, n=12). '
           'The post-cutoff bucket has 0 hawkish meetings by construction '
           '(the Fed has been on hold or cutting since July 2023). '
           "Baseline's 'unknown' column is 12/12 refusals. "
           "BM25's only error is one dovish meeting predicted as neutral.")

styled2 = (confusion.style
           .hide(axis='index')
           .set_caption(caption))
styled2

## 6. Pipeline diagram
The pipeline diagram is a hand-drawn schematic; not regenerated here.

## 7. Sanity check — headline numbers
Re-derive each headline number directly from the CSVs and compare against the expected reference values for this eval run. Prints `MATCH` or `MISMATCH` per check so anyone re-running the notebook can confirm the underlying CSVs haven't drifted.

In [ ]:
def fetch(mode, era, col):
    row = summary[(summary['mode'] == mode) & (summary['training_era'] == era)].iloc[0]
    return float(row[col])

checks = [
    ('Baseline pre-cutoff accuracy',  0.731, fetch('baseline',      'pre_cutoff',  'accuracy'), 1e-3),
    ('Baseline post-cutoff accuracy', 0.000, fetch('baseline',      'post_cutoff', 'accuracy'), 1e-3),
    ('BM25 post-cutoff accuracy',     0.917, fetch('bm25_only',     'post_cutoff', 'accuracy'), 1e-3),
    ('BM25 post-cutoff hit@5',        0.92,  fetch('bm25_only',     'post_cutoff', 'hit_at_5'), 5e-3),
    ('Hybrid post-cutoff accuracy',   0.750, fetch('hybrid',        'post_cutoff', 'accuracy'), 1e-3),
    ('Hybrid post-cutoff hit@5',      0.75,  fetch('hybrid',        'post_cutoff', 'hit_at_5'), 5e-3),
    ('Semantic post-cutoff accuracy', 0.583, fetch('semantic_only', 'post_cutoff', 'accuracy'), 1e-3),
    ('Semantic post-cutoff hit@5',    0.000, fetch('semantic_only', 'post_cutoff', 'hit_at_5'), 1e-3),
]

name_w = max(len(c[0]) for c in checks)
print(f"{'Check'.ljust(name_w)}   {'Expected':>9}   {'Computed':>9}   Status")
print('-' * (name_w + 40))
all_ok = True
for name, expected, computed, tol in checks:
    ok = abs(expected - computed) <= tol
    all_ok &= ok
    status = 'MATCH' if ok else 'MISMATCH'
    print(f'{name.ljust(name_w)}   {expected:>9.3f}   {computed:>9.4f}   {status}')

print()
print('All checks passed.' if all_ok else 'One or more checks failed — investigate the CSV inputs.')